# Auditoria oficial do eleitorado e da PNAD — apoio ao simulador da Unidade 05

**Versão 0.3 · 15 de outubro de 2026**

O objetivo é obter e examinar **dados oficiais** antes de criar a população sintética de 50 mil (piloto) e de 1 milhão de eleitores. Este notebook não gera números eleitorais inventados nem simula institutos nesta etapa.

**Fonte TSE:** [Eleitorado 2026](https://dadosabertos.tse.jus.br/dataset/eleitorado-2026). **Fonte IBGE:** [PNAD Contínua trimestral — 2026](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2026/).

O roteiro do TSE funciona apenas com Python. O roteiro da PNAD utiliza R e requer configuração adicional; há uma seção opcional ao final. Se a rede do Colab estiver indisponível, faça o download dos arquivos oficiais e envie-os manualmente ao ambiente.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, zipfile, json
import pandas as pd
from IPython.display import display

BASE = Path('/content/auditoria_eleitoral') if 'google.colab' in sys.modules else Path.cwd()/'auditoria_eleitoral'
BASE.mkdir(parents=True, exist_ok=True)
os.chdir(BASE)
print('Pasta de trabalho:', BASE)

In [ ]:
# Programas da auditoria incluídos no próprio notebook (não precisam vir do GitHub).
from pathlib import Path
Path('auditar_tse_eleitorado.py').write_text('#!/usr/bin/env python3\n"""Auditoria reproduzível do CSV agregado perfil_eleitorado_2026 do TSE.\n\nEste programa NUNCA cria percentuais de eleitorado por hipótese.\nDepende do ZIP oficial previamente baixado. Produz marginais e conjunta\nnas categorias ORIGINAIS do TSE, preservando ausência de categorias.\n\nExecutar: python auditar_tse_eleitorado.py caminho/zip_tse.zip --saida resultados_tse\n"""\nfrom __future__ import annotations\nimport argparse\nfrom collections import defaultdict\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport sys\nimport unicodedata\nimport zipfile\nimport pandas as pd\n\nCOLUNAS = {\n    \'uf\': [\'SG_UF\', \'UF\'],\n    \'sexo_registrado\': [\'DS_GENERO\', \'DS_SEXO\', \'GENERO\', \'SEXO\'],\n    \'faixa_etaria_tse\': [\'DS_FAIXA_ETARIA\', \'FAIXA_ETARIA\'],\n    \'escolaridade_tse\': [\'DS_GRAU_ESCOLARIDADE\',\'DS_GRAU_INSTRUCAO\',\'DS_ESCOLARIDADE\'],\n    \'municipio_id\': [\'CD_MUNICIPIO\',\'CD_MUN_TSE\',\'NR_MUNICIPIO\'],\n    \'municipio_nome\': [\'NM_MUNICIPIO\'],\n    \'quantidade\': [\'QT_ELEITORES_PERFIL\',\'QT_ELEITORES\',\'QT_ELEITOR\'],\n}\nOBRIGATORIAS = (\'uf\',\'sexo_registrado\',\'faixa_etaria_tse\',\'escolaridade_tse\',\'quantidade\')\n\n\ndef canonical(s: str) -> str:\n    z=unicodedata.normalize(\'NFKD\',str(s))\n    return \'\'.join(x for x in z if not unicodedata.combining(x)).strip().upper().replace(\' \',\'_\')\n\n\ndef sha256_file(p: Path)->str:\n    h=hashlib.sha256()\n    with p.open(\'rb\') as f:\n        for piece in iter(lambda:f.read(1024*1024),b\'\'):\n            h.update(piece)\n    return h.hexdigest()\n\n\ndef member(z: zipfile.ZipFile)->str:\n    files=[x for x in z.infolist() if not x.is_dir() and x.filename.lower().endswith(\'.csv\')]\n    if not files:\n        raise ValueError(\'ZIP não contém arquivo CSV; inspecionar estrutura oficial.\')\n    ranked=sorted(files,key=lambda f:((\'perfil_eleitorado\' in f.filename.lower()),f.file_size),reverse=True)\n    return ranked[0].filename\n\n\ndef processar(arquivo:Path, saida:Path, nrows_test:int|None=None):\n    if not zipfile.is_zipfile(arquivo):\n        raise ValueError(\'Arquivo de entrada não é ZIP válido.\')\n    saida.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(arquivo) as z:\n        arquivo_interno=member(z)\n        with z.open(arquivo_interno) as f:\n            header=f.readline()\n    # Cabeçalho oficial separado por ; com possível BOM.\n    head=header.decode(\'utf-8-sig\',errors=\'replace\')\n    sep=\';\' if head.count(\';\')>head.count(\',\') else \',\'\n    with zipfile.ZipFile(arquivo) as z:\n        with z.open(arquivo_interno) as f:\n            amostra=f.read(131072)\n        try:\n            amostra.decode(\'utf-8-sig\')\n            encoding=\'utf-8-sig\'\n        except UnicodeDecodeError:\n            encoding=\'latin-1\'\n        with z.open(arquivo_interno) as f:\n            preview=pd.read_csv(f,sep=sep,encoding=encoding,nrows=2,dtype=\'string\')\n    actual={canonical(k):k for k in preview.columns}\n    mapping={k:next((actual[v] for v in aliases if v in actual),None) for k,aliases in COLUNAS.items()}\n    faltam=[k for k in OBRIGATORIAS if mapping[k] is None]\n    schema={\'csv_interno\':arquivo_interno,\'separador\':sep,\'codificacao\':encoding,\n            \'colunas_originais\':list(preview.columns),\'mapeamento_candidato\':mapping,\n            \'colunas_obrigatorias_faltantes\':faltam}\n    (saida/\'esquema_tse_detectado.json\').write_text(json.dumps(schema,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    if faltam:\n        raise ValueError(f\'Colunas obrigatórias não identificadas {faltam}; consultar esquema_tse_detectado.json; NÃO inventar correspondências.\')\n    campos=[v for v in mapping.values() if v]\n    # O ZIP é processado em lotes; grandes planilhas não são carregadas na memória de uma vez.\n    sum_total=0; sum_zz=0; nlinhas=0; uf_totals=defaultdict(int)\n    joint=defaultdict(int); cities=defaultdict(int)\n    n_categorias_vazias=defaultdict(int)\n    with zipfile.ZipFile(arquivo) as z:\n        with z.open(arquivo_interno) as f:\n            chunks=pd.read_csv(f,sep=sep,encoding=encoding,dtype=\'string\',usecols=campos,\n                               chunksize=100000,low_memory=False,nrows=nrows_test)\n            for chunk in chunks:\n                nlinhas+=len(chunk)\n                q=pd.to_numeric(chunk[mapping[\'quantidade\']].str.replace(\'.\',\'\',regex=False),errors=\'coerce\')\n                if q.isna().any() or (q<0).any() or (q%1!=0).any():\n                    raise ValueError(\'Contagens eleitorais ausentes, negativas ou não inteiras no TSE.\')\n                chunk=chunk.copy()\n                chunk[\'_qt\']=q.astype(\'int64\')\n                chunk[\'_uf\']=chunk[mapping[\'uf\']].fillna(\'DESCONHECIDO\').str.upper().str.strip()\n                sum_total+=int(q.sum()); sum_zz+=int(chunk.loc[chunk[\'_uf\'].eq(\'ZZ\'),\'_qt\'].sum())\n                chunk=chunk.loc[~chunk[\'_uf\'].eq(\'ZZ\')]\n                for col in (\'sexo_registrado\',\'faixa_etaria_tse\',\'escolaridade_tse\'):\n                    canon=chunk[mapping[col]].fillna(\'NAO INFORMADO\').str.strip()\n                    n_categorias_vazias[col]+=int(canon.eq(\'\').sum())\n                    chunk[col]=canon.replace(\'\', \'NAO INFORMADO\')\n                chunk[\'uf\']=chunk[\'_uf\']\n                for vals,n in chunk.groupby([\'uf\',\'sexo_registrado\',\'faixa_etaria_tse\',\'escolaridade_tse\'],dropna=False)[\'_qt\'].sum().items():\n                    joint[vals]+=int(n)\n                for uf,n in chunk.groupby(\'uf\')[\'_qt\'].sum().items(): uf_totals[str(uf)]+=int(n)\n                if mapping.get(\'municipio_id\'):\n                    for (uf,mun),n in chunk.groupby([\'uf\',mapping[\'municipio_id\']],dropna=False)[\'_qt\'].sum().items():\n                        cities[(str(uf),str(mun))]+=int(n)\n    if not joint: raise ValueError(\'Nenhum eleitor doméstico encontrado.\')\n    dom_total=sum_total-sum_zz\n    joint_df=pd.DataFrame([(*vals,n) for vals,n in joint.items()],\n        columns=[\'uf\',\'sexo_registrado\',\'faixa_etaria_tse\',\'escolaridade_tse\',\'qt_eleitores\'])\n    joint_df=joint_df.sort_values([\'uf\',\'sexo_registrado\',\'faixa_etaria_tse\',\'escolaridade_tse\'])\n    assert int(joint_df.qt_eleitores.sum())==dom_total\n    joint_df.to_csv(saida/\'tse_conjunta_uf_sexo_idade_escolaridade.csv\',index=False)\n    for categoria, nome in [(\'uf\',\'tse_marginal_uf.csv\'),\n                           (\'sexo_registrado\',\'tse_marginal_sexo.csv\'),\n                           (\'faixa_etaria_tse\',\'tse_marginal_idade.csv\'),\n                           (\'escolaridade_tse\',\'tse_marginal_escolaridade.csv\')]:\n        marginal=joint_df.groupby(categoria,as_index=False)[\'qt_eleitores\'].sum()\n        marginal[\'fracao_eleitorado_domestico\']=marginal.qt_eleitores/dom_total\n        assert int(marginal.qt_eleitores.sum())==dom_total\n        marginal.to_csv(saida/nome,index=False)\n    if cities:\n        df=pd.DataFrame([(*vals,n) for vals,n in cities.items()],columns=[\'uf\',\'municipio_id_tse\',\'qt_eleitores\'])\n        df.sort_values([\'uf\',\'municipio_id_tse\']).to_csv(saida/\'tse_marginal_municipio.csv\',index=False)\n    ufs_validas=set(\'AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO\'.split())\n    uf_ok=set(uf_totals)==ufs_validas\n    report={\'fonte_url\':\'https://cdn.tse.jus.br/estatistica/sead/odsele/perfil_eleitorado/perfil_eleitorado_2026.zip\',\n      \'sha256_zip\':sha256_file(arquivo),\'tamanho_zip_bytes\':arquivo.stat().st_size,\n      \'csv_interno\':arquivo_interno,\'linhas_processadas\':nlinhas,\'total_arquivo\':sum_total,\n      \'total_exterior_zz\':sum_zz,\'total_domestico\':dom_total,\n      \'uf_encontradas\':sorted(uf_totals),\'n_uf\':len(uf_totals),\n      \'categorias_vazias\':dict(n_categorias_vazias),\n      \'status_27_ufs\':\'OK\' if uf_ok else \'REVISAR\',\n      \'status\':(\'RECORTE_INCOMPLETO_TESTE\' if nrows_test else\n                (\'TOTAIS_CONSISTENTES_PENDENTE_VALIDACAO_EXTERNA\' if uf_ok else \'REVISAR_COBERTURA_TERRITORIAL\')), \n      \'ressalva\':\'Total do arquivo é cadastral e não deve ser confundido com PNAD de residentes. Datas/filtros devem ser conferidos.\'}\n    (saida/\'relatorio_tse.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    return report\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'zip_tse\',type=Path)\n    p.add_argument(\'--saida\',type=Path,default=Path(\'resultado_auditoria_tse\'))\n    p.add_argument(\'--nrows-teste\',type=int,help=\'Somente testes de engenharia: não use os totais como oficiais\')\n    args=p.parse_args()\n    report=processar(args.zip_tse,args.saida,args.nrows_teste)\n    print(json.dumps(report,ensure_ascii=False,indent=2))\n\nif __name__==\'__main__\':main()\n', encoding='utf-8')
Path('validar_pnad_exportada.py').write_text('#!/usr/bin/env python3\n"""Valida uma exportação ponderada da PNAD produzida em R.\n\nNão substitui a extração oficial. Usa somente linhas do CSV de entrada e\nreporta categorias originais, faltantes e falta de suporte de células.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport pandas as pd\nimport numpy as np\n\nREQUIRED = [\'UF\',\'V2007\',\'V2009\',\'VD3005\',\'peso_pnad\']\nDONOR = [\'UF\',\'V2007\',\'V2009\',\'VD3005\',\'V1022\',\'VD4001\',\'VD4002\',\'VD4020\']\n\ndef run(fonte:Path, pasta:Path)->dict:\n    pasta.mkdir(parents=True,exist_ok=True)\n    if not fonte.exists(): raise FileNotFoundError(fonte)\n    h=hashlib.sha256()\n    with fonte.open(\'rb\') as handle:\n        for b in iter(lambda:handle.read(1<<20),b\'\'): h.update(b)\n    # Para ~300 mil registros x 9 colunas cabe em RAM do Colab.\n    df=pd.read_csv(fonte,dtype={k:\'string\' for k in DONOR},low_memory=False)\n    missing=[x for x in REQUIRED if x not in df.columns]\n    if missing: raise ValueError(f\'Colunas obrigatórias ausentes: {missing}\')\n    pesos=pd.to_numeric(df[\'peso_pnad\'],errors=\'coerce\')\n    age=pd.to_numeric(df[\'V2009\'],errors=\'coerce\')\n    if len(df)==0 or pesos.isna().any() or not np.isfinite(pesos).all() or (pesos<=0).any():\n        raise ValueError(\'Pesos doadores inválidos\')\n    if age.isna().any() or (age<16).any():\n        raise ValueError(\'Idades inválidas; esperado >= 16 (doadores adultos aptos em idade)\')\n    df[\'idade_faixa_provisoria\']=pd.cut(age,bins=[15,17,24,34,44,59,69,np.inf],labels=[\'16-17\',\'18-24\',\'25-34\',\'35-44\',\'45-59\',\'60-69\',\'70+\'])\n    df[\'_peso\']=pesos\n    columns=[\'UF\',\'V2007\',\'idade_faixa_provisoria\',\'VD3005\']\n    for col in columns:\n        df[col]=df[col].astype(\'string\').fillna(\'NAO INFORMADO\').str.strip().replace(\'\', \'NAO INFORMADO\')\n    out=df.groupby(columns,dropna=False,observed=True).agg(n_doadores=(\'UF\',\'size\'),massa_ponderada_pnad=(\'_peso\',\'sum\')).reset_index()\n    out.to_csv(pasta/\'pnad_conjunta_uf_sexo_idade_escolaridade.csv\',index=False)\n    for col in columns:\n        t=df.groupby(col,dropna=False,observed=True).agg(n_doadores=(\'UF\',\'size\'),massa_ponderada_pnad=(\'_peso\',\'sum\')).reset_index()\n        t.to_csv(pasta/f\'pnad_marginal_{col}.csv\',index=False)\n    faltantes={col:int(df[col].isna().sum()) for col in REQUIRED if col!=\'peso_pnad\'}\n    categorias={col:sorted(df[col].dropna().unique().tolist()) for col in [\'UF\',\'V2007\',\'VD3005\']}\n    report={\'status\':\'EXPORTACAO_AUDITADA_SEM_HARMONIZACAO\',\'sha256_csv\':h.hexdigest(),\n        \'arquivo\':fonte.name,\'numero_doadores_16mais\':len(df),\n        \'peso_soma\':float(pesos.sum()),\'uf_distintas\':df.UF.nunique(),\n        \'celulas_conjuntas\':len(out),\'celulas_com_1_doador\':int(out.n_doadores.eq(1).sum()),\n        \'celulas_com_menos_30_doadores\':int(out.n_doadores.lt(30).sum()),\n        \'nulos\':faltantes,\'categorias_originais\':categorias,\n        \'alerta\':\'Fonte representa moradores >=16 da PNAD, NÃO eleitorado. Categoria V2007 não igualada automaticamente à classificação cadastral do TSE. A PNAD original tem desenho complexo; tabelas de massa são estimativas pontuais.\'}\n    (pasta/\'relatorio_pnad.json\').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    return report\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'csv\',type=Path)\n    p.add_argument(\'--saida\',type=Path,default=Path(\'resultados_pnad\'))\n    args=p.parse_args()\n    print(json.dumps(run(args.csv,args.saida),indent=2,ensure_ascii=False))\n', encoding='utf-8')
Path('confrontar_territorio.py').write_text('#!/usr/bin/env python3\n"""Comparação descritiva de UFs; não equipara populações IBGE/TSE."""\nfrom pathlib import Path\nimport pandas as pd\nimport argparse\nimport json\n\nIBGE_UF={\'11\':\'RO\',\'12\':\'AC\',\'13\':\'AM\',\'14\':\'RR\',\'15\':\'PA\',\'16\':\'AP\',\'17\':\'TO\',\n         \'21\':\'MA\',\'22\':\'PI\',\'23\':\'CE\',\'24\':\'RN\',\'25\':\'PB\',\'26\':\'PE\',\'27\':\'AL\',\'28\':\'SE\',\'29\':\'BA\',\n         \'31\':\'MG\',\'32\':\'ES\',\'33\':\'RJ\',\'35\':\'SP\',\'41\':\'PR\',\'42\':\'SC\',\'43\':\'RS\',\n         \'50\':\'MS\',\'51\':\'MT\',\'52\':\'GO\',\'53\':\'DF\'}\n\ndef run(tse_dir:Path,pnad_dir:Path,out:Path):\n out.mkdir(exist_ok=True,parents=True)\n t=pd.read_csv(tse_dir/\'tse_marginal_uf.csv\',dtype={\'uf\':\'string\'})\n p=pd.read_csv(pnad_dir/\'pnad_marginal_UF.csv\',dtype={\'UF\':\'string\'})\n p[\'uf\']=p.UF.map(IBGE_UF)\n if p[\'uf\'].isna().any(): raise ValueError(f\'Código UF PNAD fora da lista IBGE: {p.loc[p.uf.isna(), "UF"].unique().tolist()}\')\n t=t.rename(columns={\'qt_eleitores\':\'eleitores_tse\'})[[\'uf\',\'eleitores_tse\']]\n p=p.rename(columns={\'n_doadores\':\'doadores_pnad\',\'massa_ponderada_pnad\':\'massa_pnad\'})[[\'uf\',\'doadores_pnad\',\'massa_pnad\']]\n j=t.merge(p,how=\'outer\',on=\'uf\',indicator=True).sort_values(\'uf\')\n if set(j[\'_merge\']) != {\'both\'}: raise ValueError(\'UF sem correspondência TSE–PNAD\')\n j[\'percentual_tse\']=100*j.eleitores_tse/j.eleitores_tse.sum()\n j[\'percentual_pnad\']=100*j.massa_pnad/j.massa_pnad.sum()\n j[\'diferenca_percentual_pnad_menos_tse\']=j.percentual_pnad-j.percentual_tse\n j.drop(columns=[\'_merge\']).to_csv(out/\'diagnostico_uf_tse_pnad.csv\',index=False)\n report={\'status\':\'DIAGNOSTICO_DESCRITIVO_NAO_CALIBRADO\',\'uf_comuns\':len(j),\n \'ufs_com_menos_30_doadores\':j.loc[j.doadores_pnad.lt(30),\'uf\'].tolist(),\n \'nota\':\'PNAD 16+ residentes ≠ eleitorado doméstico: diferenças não devem ser tratadas como erro de nenhum órgão.\'}\n (out/\'relatorio_territorio.json\').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding=\'utf8\')\n return report\nif __name__==\'__main__\':\n a=argparse.ArgumentParser();a.add_argument(\'pasta_tse\',type=Path);a.add_argument(\'pasta_pnad\',type=Path);a.add_argument(\'--saida\',type=Path,default=Path(\'resultado_confronto\'))\n args=a.parse_args();print(run(args.pasta_tse,args.pasta_pnad,args.saida))\n', encoding='utf-8')
Path('gerar_templates_harmonizacao.py').write_text('#!/usr/bin/env python3\n"""Produz planilhas de harmonização NÃO preenchidas a partir de categorias reais."""\nimport argparse\nfrom pathlib import Path\nimport pandas as pd\n\nPAIR = [(\'sexo_registrado\',\'V2007\',\'sexo\'),(\'escolaridade_tse\',\'VD3005\',\'escolaridade\'),(\'faixa_etaria_tse\',\'idade_faixa_provisoria\',\'idade\')]\n\ndef run(tse_dir: Path, pnad_dir: Path, out: Path):\n    t=pd.read_csv(tse_dir/\'tse_conjunta_uf_sexo_idade_escolaridade.csv\',dtype=\'string\')\n    p=pd.read_csv(pnad_dir/\'pnad_conjunta_uf_sexo_idade_escolaridade.csv\',dtype=\'string\')\n    out.mkdir(parents=True,exist_ok=True)\n    summary=[]\n    for src_t,src_p,kind in PAIR:\n        a=sorted(t[src_t].fillna(\'NAO INFORMADO\').unique())\n        b=sorted(p[src_p].fillna(\'NAO INFORMADO\').unique())\n        records=([{\'fonte\':\'TSE\',\'variavel\':src_t,\'categoria_original\':v,\'categoria_harmonizada\':\'\',\'justificativa\':\'\',\'aprovado\':\'\'} for v in a]\n            + [{\'fonte\':\'PNAD\',\'variavel\':src_p,\'categoria_original\':v,\'categoria_harmonizada\':\'\',\'justificativa\':\'\',\'aprovado\':\'\'} for v in b])\n        pd.DataFrame(records).to_csv(out/f\'crosswalk_{kind}_PREENCHER.csv\',index=False)\n        summary.append({\'dimensao\':kind,\'categorias_tse\':len(a),\'categorias_pnad\':len(b),\'status\':\'PENDENTE_REVISAO_HUMANA\'})\n    pd.DataFrame(summary).to_csv(out/\'status_harmonizacao.csv\',index=False)\n    return summary\nif __name__==\'__main__\':\n    q=argparse.ArgumentParser()\n    q.add_argument(\'pasta_tse\',type=Path)\n    q.add_argument(\'pasta_pnad\',type=Path)\n    q.add_argument(\'--saida\',type=Path,default=Path(\'harmonizacao\'))\n    args=q.parse_args();print(run(args.pasta_tse,args.pasta_pnad,args.saida))\n', encoding='utf-8')
Path('extrair_pnad_2026T2.R').write_text('# Extração proposta da PNAD Contínua 2º trimestre de 2026.\n# Precisa de conexão à internet e dos pacotes R PNADcIBGE, survey, utils.\n# A execução com microdados oficiais AINDA NÃO foi realizada neste ambiente.\n# Saída preserva códigos originais e pesos; não usa percentuais inventados.\n\nif (!requireNamespace("PNADcIBGE", quietly=TRUE)) stop("Instale PNADcIBGE")\nif (!requireNamespace("survey", quietly=TRUE)) stop("Instale survey")\n\nout <- "resultado_auditoria_pnad"\ndir.create(out, showWarnings=FALSE, recursive=TRUE)\n# As variáveis escolhidas devem ser conferidas com o dicionário vigente.\nvars <- c("UF", "V1022", "V2007", "V2009", "VD3005", "VD4001", "VD4002", "VD4020")\n\ndir.create(file.path(out,"downloads"), showWarnings=FALSE, recursive=TRUE)\ncat("Obtendo PNAD 2º tri 2026 com desenho amostral...\\n")\ndes <- PNADcIBGE::get_pnadc(\n  year=2026, quarter=2, vars=vars,\n  labels=FALSE, deflator=FALSE, design=TRUE,\n  reload=FALSE, savedir=file.path(out,"downloads")\n)\nif (is.null(des)) stop("PNAD não retornou dados")\nbase <- des$variables\nnecessarias <- c("UF","V2007","V2009","VD3005")\nfaltam <- setdiff(necessarias, names(base))\nif (length(faltam)>0) stop(paste("Variáveis ausentes no microdado:", paste(faltam,collapse=",")))\n\n\n# PNAD contém residentes, NÃO define quem é eleitor apto.\n# Idade >=16 apenas restringe registros doadores; aplicar no desenho, não só nos valores.\n# O filtro preserva os pesos e a estrutura de amostragem do objeto survey.\ndes_16 <- subset(des, !is.na(V2009) & as.numeric(as.character(V2009))>=16)\nbase <- des_16$variables\nbase$peso_pnad <- as.numeric(survey::weights(des_16, type="sampling"))\nif (anyNA(base$peso_pnad) || any(base$peso_pnad<=0)) stop("Pesos PNAD inválidos")\n\nvars_presentes <- intersect(c(vars,"peso_pnad"), names(base))\n# As variáveis categóricas devem permanecer como códigos sem conversão automática.\n# A coluna peso_pnad conserva o peso de expansão das observações doadoras.\nwrite.csv(base[,vars_presentes,drop=FALSE], file.path(out,"pnad_doadores_codigos_originais.csv"),row.names=FALSE,fileEncoding="UTF-8")\n\n# Agregação simples de controles de massa: usa pesos do desenho para estimativas pontuais.\n# A estimação de erros-padrão requer o objeto completo de desenho (salvo em RDS).\nbase$faixa_idade_piloto <- cut(as.numeric(as.character(base$V2009)),\n    breaks=c(15,17,24,34,44,59,69,Inf),\n    labels=c("16-17","18-24","25-34","35-44","45-59","60-69","70+"),\n    right=TRUE)\n# Faixas provisórias, não são automaticamente equivalentes às faixas TSE.\ncampos_joint <- intersect(c("UF","V2007","faixa_idade_piloto","VD3005"),names(base))\nconjunta <- aggregate(base$peso_pnad, by=base[campos_joint], FUN=sum)\nnames(conjunta)[ncol(conjunta)] <- "massa_ponderada_pnad"\nwrite.csv(conjunta,file.path(out,"pnad_conjunta_uf_sexo_escolaridade.csv"),row.names=FALSE,fileEncoding="UTF-8")\nsaveRDS(des_16, file.path(out,"pnad_desenho_16mais.rds"))\ncat("Arquivos emitidos em:", out, "\\n")\n', encoding='utf-8')
print('Programas preparados:', [p.name for p in Path.cwd().glob('*.py')])

## Parte A — Perfil do eleitorado oficial (TSE)

Execute a célula de download. Ela procura primeiro o ZIP já existente e só tenta obter a fonte se necessário. Depois, os programas verificam as colunas, excluem o eleitorado no exterior (`ZZ`) e produzem marginais e cruzamentos sem inventar categorias. O tamanho e o SHA-256 serão registrados.

**Atenção:** a interpretação só deve avançar se o relatório identificar todas as 27 UFs e se os totais forem compatíveis com a documentação oficial. Consistência das somas, sozinha, não comprova a correção externa das categorias.

In [ ]:
from urllib.request import urlretrieve
FONTE_TSE = 'https://cdn.tse.jus.br/estatistica/sead/odsele/perfil_eleitorado/perfil_eleitorado_2026.zip'
ZIP_TSE = BASE/'fontes_brutas'/'perfil_eleitorado_2026.zip'
ZIP_TSE.parent.mkdir(exist_ok=True)
if not (ZIP_TSE.exists() and zipfile.is_zipfile(ZIP_TSE)):
    try:
        urlretrieve(FONTE_TSE, ZIP_TSE)
    except Exception as e:
        print('Falha ao obter o ZIP oficial:', repr(e))
        print('Baixe o arquivo no portal do TSE e coloque-o em:', ZIP_TSE)
        raise
assert zipfile.is_zipfile(ZIP_TSE), 'ZIP TSE inválido.'
print('ZIP TSE disponível:', ZIP_TSE, 'bytes:', ZIP_TSE.stat().st_size)

In [ ]:
subprocess.run([
    sys.executable, 'auditar_tse_eleitorado.py', str(ZIP_TSE),
    '--saida', 'resultados_tse'
], check=True)
rel_tse = json.loads(Path('resultados_tse/relatorio_tse.json').read_text(encoding='utf8'))
print('Status:', rel_tse['status'])
print('Eleitores domésticos:', rel_tse['total_domestico'])
print('Exterior excluído:', rel_tse['total_exterior_zz'])
print('UFs encontradas:', rel_tse['n_uf'])
display(pd.read_csv('resultados_tse/tse_marginal_uf.csv').sort_values('qt_eleitores',ascending=False).head(27))

## Parte B — PNAD Contínua 2026, segundo trimestre

A PNAD não é o cadastro de eleitores: os doadores serão moradores de **16 anos ou mais**, selecionados da base de domicílios com os pesos originais. Escolaridade declarada e escolaridade cadastral do TSE não podem ser igualadas automaticamente.

O script R `extrair_pnad_2026T2.R` está salvo na pasta de trabalho. Ele precisa de `Rscript` e das bibliotecas `PNADcIBGE` e `survey`. A fonte oficial é grande (o ZIP publicado tem cerca de 223 MB), e instalar/baixar pode levar tempo.

A próxima célula informa se R está disponível. A célula opcional seguinte poderá instalar R, PNADcIBGE e survey e executar a extração, **somente se você alterar** `EXECUTAR_PNAD_NO_COLAB = True`. Esta instalação pode consumir bastante tempo e armazenamento. Em Linux local com R instalado, execute `Rscript extrair_pnad_2026T2.R`. O arquivo de doadores será grande e deve ficar fora do GitHub.

In [ ]:
print('Rscript disponível:', shutil.which('Rscript'))
print('Script:', BASE/'extrair_pnad_2026T2.R')
print('Execute em terminal R configurado: Rscript extrair_pnad_2026T2.R')

In [ ]:
# OPCIONAL: defina True para instalar/executar R no Colab.
# Esta etapa baixa centenas de MB e pode demorar bastante.
EXECUTAR_PNAD_NO_COLAB = False

if EXECUTAR_PNAD_NO_COLAB:
    if shutil.which('Rscript') is None:
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run([
            'apt-get','install','-y','-qq', 'r-base',
            'libcurl4-openssl-dev','libssl-dev','libxml2-dev'
        ], check=True)
    subprocess.run([
        'Rscript','-e',
        "options(repos=c(CRAN='https://cloud.r-project.org')); "
        "install.packages(c('PNADcIBGE','survey'), Ncpus=2)"
    ], check=True)
    subprocess.run(['Rscript','extrair_pnad_2026T2.R'], check=True)
else:
    print('Etapa PNAD não executada. Para habilitar, ajuste EXECUTAR_PNAD_NO_COLAB=True.')


### Validar os doadores PNAD e gerar as correspondências

Quando `resultado_auditoria_pnad/pnad_doadores_codigos_originais.csv` estiver disponível, execute a célula seguinte. Ela calcula tabelas ponderadas, identifica categorias e amostras escassas, compara os perfis territoriais entre fontes e cria **templates vazios** para harmonizar sexo, escolaridade e idade.

**Não aprove a população-piloto antes de examinar essas correspondências com os dicionários do TSE e IBGE.**

In [ ]:
CSV_PNAD = BASE/'resultado_auditoria_pnad'/'pnad_doadores_codigos_originais.csv'
if not CSV_PNAD.exists():
    print('PENDENTE — execute primeiro o extrator R da PNAD; arquivo esperado:', CSV_PNAD)
else:
    for command in [
        [sys.executable, 'validar_pnad_exportada.py', str(CSV_PNAD), '--saida', 'resultados_pnad'],
        [sys.executable, 'confrontar_territorio.py', 'resultados_tse', 'resultados_pnad', '--saida', 'confrontacao'],
        [sys.executable, 'gerar_templates_harmonizacao.py', 'resultados_tse', 'resultados_pnad', '--saida', 'harmonizacao'],
    ]:
        subprocess.run(command, check=True)
    display(pd.read_csv('confrontacao/diagnostico_uf_tse_pnad.csv').head())
    print('Templates gerados em:', BASE/'harmonizacao')

## Parte C — Guardar e compartilhar o relatório de auditoria

O ZIP final contém as **tabelas agregadas e os relatórios disponíveis**. Não inclui o ZIP bruto oficial do TSE nem os microdados PNAD. Caso a extração da PNAD ainda não tenha ocorrido, o pacote refletirá somente os resultados do TSE, sem falsificar uma etapa concluída.

In [ ]:
import zipfile
PACOTE = BASE/'relatorios_auditoria_fontes.zip'
with zipfile.ZipFile(PACOTE,'w',zipfile.ZIP_DEFLATED) as z:
    for pasta in ['resultados_tse','resultados_pnad','confrontacao','harmonizacao']:
        origem=BASE/pasta
        if origem.exists():
            for p in origem.rglob('*'):
                if p.is_file(): z.write(p,p.relative_to(BASE))
print('Relatórios consolidados:', PACOTE)
try:
    from google.colab import files
    files.download(str(PACOTE))
except ImportError:
    print('Em ambiente local, o arquivo permanece em:', PACOTE)